In [ ]:
# import modules to be used in this notebook
#   numpy is a math library, mercury is an app library for jupyter notebooks
import numpy as np
import mercury as mr

# for general plots, including static plots
import matplotlib.pyplot as plt

# for animated plots
import matplotlib.animation as animation
from IPython import display

# initiate web-app with mercury
app = mr.App(title="Diffusion on Lattice", description="Random walk model", show_code=False)

### Random walk in 3D

In [ ]:
# Random walk on 3D lattice

import numpy as np

def random_walk(nt,latt_type):
    """
    Random walk on a 3D lattice
    Inputs:
        nt [integer] = number of desired jumps (i.e., time steps)
        latt_type [string] = choose the lattice geometry
                             current options: "cubic" and "fcc"
    Outputs:
        rs2 [array] = square displacement at each time step
        x   [array] = x-coordinate at each time step
        y   [array] = y-coordinate at each time step
        z   [array] = z-coordinate at each time step
    """
    x = np.zeros(nt+1)
    y = np.zeros(nt+1)
    z = np.zeros(nt+1)
    rs2 = np.zeros(nt+1)

    x[0] = 0
    y[0] = 0
    z[0] = 0
    rs2[0] = 0

    if latt_type == "cubic":
        # cubic lattice
        fd = np.floor(6 * np.random.rand(nt))
        delx = np.array([1, -1, 0, 0, 0, 0])
        dely = np.array([0, 0, 1, -1, 0, 0])
        delz = np.array([0, 0, 0, 0, 1, -1])
    elif latt_type == "fcc":
        # fcc lattice- 12 nearest neighbors
        fd = np.floor(12.* np.random.rand(nt)); # list with nt entries, 12 nearest neighbors
        delx = 0.5*np.array([1, 1, 0, -1, -1, -1, 0, 1, 0, -1, 1, 0]); # these three lines define the jumps on the FCC lattice
        dely = 0.5*np.array([1, 0, 1, 1, 0, 0, 1, 0, -1, -1, -1, -1]);
        delz = 0.5*np.array([0, 1, 1, 0, -1, 1, -1, -1, 1, 0, 0, -1]); 
    else:
    	raise ValueError("Lattice type not implemented! See lattice_3d.py")

    #sum over nt jumps
    for j in range(nt):
        x[j+1] = x[j] + delx[int(fd[j]) - 1] # x position at j+1 jump
        y[j+1] = y[j] + dely[int(fd[j]) - 1] # y position at j+1 jump
        z[j+1] = z[j] + delz[int(fd[j]) - 1] # z position at j+1 jump

        # square displacement position at j+1 jump in 3D
        rs2[j+1] = x[j+1]**2 + y[j+1]**2 + z[j+1]**2

    return rs2, x, y, z


In [ ]:
# Run random walk on a 2D lattice
# add Mercury widgets
lattice = mr.Select(label="Choose a 3D lattice: ", value="cubic",choices=["cubic","face-centered cubic"])
num_steps = mr.Numeric(value=1e2,min=0,max=1e3,label="Enter the number of steps in the simulation:")
disp = mr.Select(label="Choose a display type:", value="static (fast)", choices=["static (fast)","interactive (slow)"])

In [ ]:
# convert float to integer
nsteps = int(num_steps.value)
# run the random walk simulation
rs2, x, y, z = random_walk(nsteps,latt_type=lattice.value)

In [ ]:
## display the results
if disp.value == "static (fast)":
    ## a  static plot of the results
    label = 'Steps = {}'.format(nsteps)
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    plt.title(f"Random walk trajectory on {lattice.value} lattice with {nsteps} steps")
    ax.grid(True)
    ax.legend([label], loc='upper left', bbox_to_anchor=(1, 1), borderaxespad=0.)
    ax.set_xlabel('x position')
    ax.set_ylabel('y position')
    ax.set_zlabel('z position')
    ax.plot(x, y, z)
else:
    ## an animated plot of the results
    # create line initially without data
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    line = ax.plot([],[],[])[0]

    # define a function that will update the results frame-by-frame
    def update_3D(frame_num, line, x, y, z):
        line.set_data(x[:frame_num],y[:frame_num]) ## WW: a bug here? interactive differs from static visualization
        line.set_3d_properties(z[:frame_num])

    # Setting the axes properties
    ax.set(xlim3d=(min(x),max(x)), xlabel='X')
    ax.set(ylim3d=(min(y),max(y)), ylabel='Y')
    ax.set(ylim3d=(min(z),max(z)), ylabel='Z')
    
    # Creating the Animation object
    #   interval = delay 
    ani = animation.FuncAnimation(fig, update_3D, nsteps, fargs=(line,x,y,z),interval=50)
    
    ## embedded video
    #video = ani.to_html5_video()
    #html = display.HTML(video)
    #display.display(html)
    #plt.close()
    
    ## interactive video
    video = ani.to_jshtml()
    html = display.HTML(video)
    display.display(html)
    plt.close()